## Reconocimiento de imágenes: Ajuste de una red convolucional para datos de dígitos 

### Empezamos ajustando con un perceptrón multicapa usual 

In [2]:
'''Trains a simple deep NN on the MNIST dataset.

Gets to 98.40% test accuracy after 20 epochs
(there is *a lot* of margin for parameter tuning).
2 seconds per epoch on a K520 GPU.
'''

from __future__ import print_function

import keras
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten
from keras.optimizers import RMSprop
from keras.layers import Conv2D, MaxPooling2D

#Tmaño del batch, el output son 10 clases y 20 epochs

batch_size = 128
num_classes = 10
epochs = 20

# the data, shuffled and split between train and test sets
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Todo esto lo hemos hecho antes, tomar los datos de entrenamiento (28*28 pixeles, de allí el 784), dividir
#lo de la escala de grisis entre 255
x_train = x_train.reshape(60000, 784)
x_test = x_test.reshape(10000, 784)
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')

# convert class vectors to binary class matrices. Aquí es necesario porque no decimos explícito que
#en la última capa hay 10 nodos
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

#Please construct the following neural network and report accuracy after training
#Layer 1: 2D Convolution with 32 hidden neurons, kernel 3 by 3, relu activation, input_shape (28,28,1)
#Layer 2: 2D MaxPooling, pool_size 2 by 2
#Layer 3: Flatten (Hint: model.add(Flatten()))
#Layer 4 Softmax Output Layer according to the problem (output vector)

#La red multicapa para este caso tiene dos capas ocultas, cada una con 512 nodos, partimos
#de los 784 pixeles (inputs) y se desechan 20% de las ligas entre capas

model = Sequential()

model.add(Dense(512, activation='relu', input_shape=(784,)))
model.add(Dropout(0.2))
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(num_classes, activation='softmax'))

model.summary()

#Se usa cross entropy para definir la función de perdida 

model.compile(loss='categorical_crossentropy',
              optimizer=RMSprop(),
              metrics=['accuracy'])


#some learners constantly reported 502 errors in Watson Studio. 
#This is due to the limited resources in the free tier and the heavy resource consumption of Keras.
#This is a workaround to limit resource consumption

#from keras import backend as K

#K.set_session(K.tf.Session(config=K.tf.ConfigProto(intra_op_parallelism_threads=1, inter_op_parallelism_threads=1)))

history = model.fit(x_train, y_train,
                    batch_size=batch_size,
                    epochs=epochs,
                    verbose=1,
                    validation_data=(x_test, y_test))
score = model.evaluate(x_test, y_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])


60000 train samples
10000 test samples
Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 512)               401920    
                                                                 
 dropout_2 (Dropout)         (None, 512)               0         
                                                                 
 dense_4 (Dense)             (None, 512)               262656    
                                                                 
 dropout_3 (Dropout)         (None, 512)               0         
                                                                 
 dense_5 (Dense)             (None, 10)                5130      
                                                                 
Total params: 669,706
Trainable params: 669,706
Non-trainable params: 0
_________________________________________________________________
Epoch 1/2

### Ahora con la red convolucional

In [5]:
#Tmaño del batch, el output son 10 clases y 20 epochs

batch_size = 128
num_classes = 10
epochs = 20

# the data, shuffled and split between train and test sets
(x_train, y_train), (x_test, y_test) = mnist.load_data()


#Esto es similar a lo de  arriba, el cambio es en la dimensión ya no son los elemento de 28*28 pixeles
#como columnas sino que tenemos las matrices de 28*28 por separado en cada una de las observaciones
#6000 para entrenamiento, lo que corresponde a x_train.shape[0] y similar en el de prueba

x_train = x_train.reshape((x_train.shape[0], 28, 28, 1))
x_test = x_test.reshape((x_test.shape[0], 28, 28, 1))

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')

# convert class vectors to binary class matrices. Aquí es necesario porque no decimos explícito que
#en la última capa hay 10 nodos
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

#Please construct the following neural network and report accuracy after training

#Layer 1: 2D Convolution with 32 hidden neurons, kernel 3 by 3, relu activation, input_shape (28,28,1)
#Se aplica primero segun input shape (para uno o más canales) una convolución
#2D que corresponde a aplicar un operador por cada malla de celdas de 3*3 pixeles (kernel size) y 32 filtros,
#aspectos de la imagen a puntualizar según la convolución.

#Layer 2: 2D MaxPooling, pool_size 2 by 2
#Se aplica una capa tipo pooling las cuaes sirven para analziar partes de la imagen, por ejemplo aquí
#los pixeles mas saturados de color

#Layer 3: Flatten (Hint: model.add(Flatten()))
##De allí se aplasta, Flatten() la red para ya considerar un formato tipo perceptron multicapa

#Layer 4 Softmax Output Layer according to the problem (output vector). Sotmax es la adecuada
#para clasificación de cada output posible (los 10 dígitos)


model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_uniform', input_shape=(28, 28, 1)))
model.add(MaxPooling2D((2, 2)))
model.add(Flatten())
model.add(Dense(10, activation='softmax'))
model.summary()

#Se usa cross entropy para definir la función de perdida 

model.compile(loss='categorical_crossentropy',
              optimizer=RMSprop(),
              metrics=['accuracy'])

#Vemos una ligera mejora en la métrica respecto al uso de simplemente una red multicapa
history = model.fit(x_train, y_train,
                    batch_size=batch_size,
                    epochs=epochs,
                    verbose=1,
                    validation_data=(x_test, y_test))
score = model.evaluate(x_test, y_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])


60000 train samples
10000 test samples
Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_1 (Conv2D)           (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 13, 13, 32)       0         
 2D)                                                             
                                                                 
 flatten_1 (Flatten)         (None, 5408)              0         
                                                                 
 dense_7 (Dense)             (None, 10)                54090     
                                                                 
Total params: 54,410
Trainable params: 54,410
Non-trainable params: 0
_________________________________________________________________
Epoch 1/20
469/469 [==============================] - 12s 24ms/step - loss: 0